# Ideas planner, second list, idea 3: deformable-part correlation over the whole disc (and the pose sweep that has to come first)
Step A, pose sweep: for the photos whose right place never reaches the candidates, run the coarse correlation alone at many poses (heading -30..+30 step 5 x scale 0.7..1.3 step 0.1) and see whether the true place enters the 200 best peaks at ANY pose. If yes, the loss is pose coverage and a new correlation is not what is needed.
Step B, parts: at the six standard poses, correlate the whole canvas, the centre disc (60 % of the width) and the four quadrants separately; give each quadrant map slack (max over a 5 x 5 cell window = +-2 cells, about 7 m at level 3) and combine the four by mean or minimum. Compare the rank of the true place in the peak list of each map.
Photos: NEVER = the photos listed in SEED_IDS_FILE (csv column photo; NUDT valid draw 1212: 9 never-retrieved; official: the 12 low-altitude skips). Controls = photos the pipeline retrieves today (the true place must not fall out).
Falsification (planned): step B is dead if fewer than 3 of the 9 never-retrieved photos get the true place into the top 50 of any map variant; step A: if the true place enters the top 50 at some pose for most photos, the cause is pose coverage.

In [ ]:
import os, sys, time, gc, functools
from pathlib import Path
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
T7 = ROOT / "tests/Test_Iteration_7/results"

DATA = os.environ.get("GATE_DATA", "official")          # official or nudt
POOL_PHOTOS = int(os.environ.get("POOL_PHOTOS", "500"))
TAG = os.environ.get("TAG", "official150")              # which rich dump (results/rich_dump/<TAG>) names the photos
N_LIMIT = int(os.environ.get("N_LIMIT", "0"))           # >0: only the first photos (smoke test)
print(DATA, TAG, "pool", POOL_PHOTOS, "limit", N_LIMIT)

In [ ]:
if DATA == "nudt":
    from train.datasets import photo_sets
    photo_sets.load_nudt_photos = functools.partial(photo_sets.load_nudt_photos, split=os.environ.get("NUDT_SPLIT", "test"))
from train.checks.rich_dump import RichDumpPipeline
from train.checks import replay
from coarse_search.heading_queries import query_geometry
from image_processing.geometry import render_query
from image_processing.map_levels import level_for_height, level_gsd

def make_pipeline(headings, scales, confidence=0.22, coarse_threshold=0.21):
    pipeline = RichDumpPipeline(DATA, "edge_zncc", semantic_layer=True, semantic_decision=True, gamma=1.0, k_peaks=50, m_fine=10,
                                matcher="loftr_minima", min_inliers=8, ransac_pixels=None, matcher_confidence=confidence, satellite="new",
                                map_margin_m=None, loftr_coarse_threshold=coarse_threshold, dino_weight=None, headings=headings, scale_factors=scales)
    start = time.time()
    pipeline.setup(POOL_PHOTOS, 7)
    print("setup %.0f s" % (time.time() - start))
    return pipeline

STANDARD = [(-5.0, 1.0), (0.0, 1.0), (5.0, 1.0), (-5.0, 1.05), (0.0, 1.05), (5.0, 1.05)]
dump = replay.load_set(T7 / "rich_dump" / TAG)

In [ ]:
pipeline = make_pipeline([-5.0, 0.0, 5.0], [1.0, 1.05])
from train.checks import part_maps
never = [int(i) for i in pd.read_csv(os.environ["SEED_IDS_FILE"]).photo]
control = [p for p in dump if p not in never][: int(os.environ.get("CONTROL_N", "10"))]
if N_LIMIT:
    never, control = never[:N_LIMIT], control[:N_LIMIT]
test_set = pipeline.test_set
print(len(never), "never retrieved;", len(control), "controls")

In [ ]:
SWEEP_HEADINGS = [float(h) for h in range(-30, 31, 5)]
SWEEP_SCALES = [0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3]
rows = []
start = time.time()
for group, ids in [("never", never), ("control", control)]:
    for pid in ids:
        item = test_set.items[pid]; prior = test_set.priors[pid]; truth = [float(v) for v in test_set.truths[pid]]; sigma = float(test_set.sigmas[pid])
        level = level_for_height(item["rel_height"], pipeline.camera)
        poses = [(h, s) for h in SWEEP_HEADINGS for s in SWEEP_SCALES] if group == "never" else STANDARD
        for h, s in poses:
            canvas, valid = render_query(item, pipeline.camera, level, query_geometry(h, s))
            score_map, geometry = part_maps.zncc_map(pipeline.store, canvas, valid, level, prior)
            rank = part_maps.truth_rank(part_maps.peaks_of(score_map, geometry), truth, sigma)
            rows.append(dict(group=group, photo=pid, height=item["rel_height"], heading=h, scale=s, standard=int((h, s) in STANDARD), rank=rank))
            del score_map, geometry
        torch.cuda.empty_cache(); gc.collect()
        print(group, pid, "%.0f s" % (time.time() - start), flush=True)
S = pd.DataFrame(rows)
S.to_csv(T7 / ("idea3_pose_sweep_%s.csv" % TAG), index=False)
def hit(r, m): return (r > 0) & (r <= m)
P = S.groupby(["group", "photo"]).apply(lambda g: pd.Series(dict(
    best_rank_standard=int(g[g.standard == 1]["rank"][g[g.standard == 1]["rank"] > 0].min()) if (g[g.standard == 1]["rank"] > 0).any() else 0,
    best_rank_any_pose=int(g["rank"][g["rank"] > 0].min()) if (g["rank"] > 0).any() else 0,
    top50_standard=int(hit(g[g.standard == 1]["rank"], 50).any()), top50_any_pose=int(hit(g["rank"], 50).any()),
    poses_in_top50=int(hit(g["rank"], 50).sum()), poses=len(g)))).reset_index()
P.to_csv(T7 / ("idea3_pose_sweep_photos_%s.csv" % TAG), index=False)
print(P.to_string(index=False))
print(P.groupby("group")[["top50_standard", "top50_any_pose"]].agg(["sum", "count"]).to_string())

In [ ]:
masks, centre = part_maps.part_masks(224 // 4)
rows = []
for group, ids in [("never", never), ("control", control)]:
    for pid in ids:
        item = test_set.items[pid]; prior = test_set.priors[pid]; truth = [float(v) for v in test_set.truths[pid]]; sigma = float(test_set.sigmas[pid])
        level = level_for_height(item["rel_height"], pipeline.camera)
        for h, s in STANDARD:
            canvas, valid = render_query(item, pipeline.camera, level, query_geometry(h, s))
            maps = {}
            full, geometry = part_maps.zncc_map(pipeline.store, canvas, valid, level, prior)
            maps["full"] = full
            maps["centre"], _ = part_maps.zncc_map(pipeline.store, canvas, valid, level, prior, centre)
            quarters = [part_maps.zncc_map(pipeline.store, canvas, valid, level, prior, m)[0] for m in masks.values()]
            if all(q is not None for q in quarters):
                for slack in (1, 5):
                    pooled = [part_maps.pool_slack(q, slack) if slack > 1 else q for q in quarters]
                    maps["parts_mean_slack%d" % slack] = part_maps.combine(pooled, "mean")
                    maps["parts_min_slack%d" % slack] = part_maps.combine(pooled, "min")
            for name, score_map in maps.items():
                if score_map is None:
                    continue
                rank = part_maps.truth_rank(part_maps.peaks_of(score_map, geometry), truth, sigma)
                rows.append(dict(group=group, photo=pid, height=item["rel_height"], heading=h, scale=s, method=name, rank=rank))
            del maps, quarters
            torch.cuda.empty_cache(); gc.collect()
        print("parts", group, pid, flush=True)
B = pd.DataFrame(rows)
B.to_csv(T7 / ("idea3_parts_%s.csv" % TAG), index=False)
best = B.groupby(["group", "photo", "method"])["rank"].apply(lambda r: int(r[r > 0].min()) if (r > 0).any() else 0).reset_index()
best["top50"] = ((best["rank"] > 0) & (best["rank"] <= 50)).astype(int)
best["top10"] = ((best["rank"] > 0) & (best["rank"] <= 10)).astype(int)
summary = best.groupby(["group", "method"])[["top10", "top50"]].sum().reset_index()
summary["photos"] = best.groupby(["group", "method"]).size().values
summary.to_csv(T7 / ("idea3_parts_summary_%s.csv" % TAG), index=False)
print(summary.to_string(index=False))